In [1]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Load Function 7 data
X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy")
print("X shape:", X.shape)
print("Y shape:", Y.shape)


print(X)


print(Y)

X shape: (30, 6)
Y shape: (30,)
[[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [

In [2]:
best_index = np.argmax(Y)

print("Best index:", best_index)
print("Best current x:", X[best_index])
print("Best current y:", Y[best_index])
print("Best current portal format:", "-".join(f"{v:.6f}" for v in X[best_index]))

kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

mean, std = gp.predict(candidates, return_std=True)

kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Best index: 6
Best current x: [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
Best current y: 1.3649683044991994
Best current portal format: 0.057896-0.491672-0.247422-0.218118-0.420428-0.730970
Suggested query: [0.14843726 0.53592728 0.27639726 0.09680857 0.34599854 0.75101258]
Portal format: 0.148437-0.535927-0.276397-0.096809-0.345999-0.751013
Predicted mean: 1.169850805662703
Predicted std: 0.17207313994780185
UCB score: 1.6000336555322077


In [3]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
#For Function 7, I used UCB because Week 1 did not improve the best value and the function is six-dimensional, making manual reasoning unreliable.
#UCB was useful here because it balanced the predicted output with uncertainty, allowing the next query to explore around the current best region without over-committing to it.
# Load original Function 7 data
X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy")

# Add Week 1 query and output
week1_x = np.array([[0.148437, 0.535927, 0.276397, 0.096809, 0.345999, 0.751013]])
week1_y = np.array([1.1745584410367518])

X = np.vstack([X, week1_x])
Y = np.append(Y, week1_y)

print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

print("Best current x:", X[np.argmax(Y)])
print("Best current y:", np.max(Y))

# Fit GP surrogate model
kernel = C(1.0) * RBF(length_scale=0.2)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    random_state=42
)

gp.fit(X, Y)

# Generate candidate points
rng = np.random.default_rng(42)
candidates = rng.uniform(0, 1, size=(10000, X.shape[1]))

# Predict mean and uncertainty
mean, std = gp.predict(candidates, return_std=True)

# UCB acquisition
kappa = 2.5
ucb = mean + kappa * std

best_ucb_index = np.argmax(ucb)
query = candidates[best_ucb_index]

print("Acquisition used: UCB")
print("Suggested query:", query)
print("Portal format:", "-".join(f"{v:.6f}" for v in query))
print("Predicted mean:", mean[best_ucb_index])
print("Predicted std:", std[best_ucb_index])
print("UCB score:", ucb[best_ucb_index])

Updated X shape: (31, 6)
Updated Y shape: (31,)
Best current x: [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
Best current y: 1.3649683044991994
Acquisition used: UCB
Suggested query: [0.03316913 0.32973803 0.3663577  0.23430088 0.28631503 0.70492743]
Portal format: 0.033169-0.329738-0.366358-0.234301-0.286315-0.704927
Predicted mean: 1.0899648162350408
Predicted std: 0.20861300288668916
UCB score: 1.6114973234517638


In [4]:
import numpy as np
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C


# For Function 7, Week 2 improved strongly over Week 1.
# Because this is a six-dimensional function, I do not want to rely only on local exploitation.
# I use a filtered hybrid EI-UCB strategy that searches around the current best point
# while still including global candidates for uncertainty-aware exploration.


# -----------------------------
# Load original Function 7 data
# -----------------------------
X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy")


# -----------------------------
# Add Week 1 and Week 2 results
# -----------------------------
week1_x = np.array([[0.148437, 0.535927, 0.276397, 0.096809, 0.345999, 0.751013]])
week1_y = np.array([1.1745584410367518])

week2_x = np.array([[0.033169, 0.329738, 0.366358, 0.234301, 0.286315, 0.704927]])
week2_y = np.array([2.4608579737515917])

X = np.vstack([X, week1_x, week2_x])
Y = np.append(Y, [week1_y[0], week2_y[0]])


print("Updated X shape:", X.shape)
print("Updated Y shape:", Y.shape)

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Best current x:", best_x)
print("Best current y:", best_y)


# -----------------------------
# Fit GP surrogate model
# -----------------------------
kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(length_scale=np.ones(X.shape[1]) * 0.2, length_scale_bounds=(1e-2, 1.0), nu=2.5)
    + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)


# -----------------------------
# Generate candidate points
# -----------------------------
rng = np.random.default_rng(42)
dim = X.shape[1]

# Six-dimensional space is harder to search, so keep a meaningful global pool.
global_candidates = rng.uniform(0, 1, size=(30000, dim))

# Week 2 improved strongly, so generate local candidates around the current best point.
local_candidates = rng.normal(loc=best_x, scale=0.12, size=(25000, dim))
local_candidates = np.clip(local_candidates, 0, 1)

candidates = np.vstack([global_candidates, local_candidates])


# -----------------------------
# Predict mean and uncertainty
# -----------------------------
mean, std = gp.predict(candidates, return_std=True)

y_best = np.max(Y)
std_safe = std + 1e-12


# -----------------------------
# Expected Improvement
# -----------------------------
improvement = mean - y_best
z = improvement / std_safe
ei = improvement * norm.cdf(z) + std_safe * norm.pdf(z)


# -----------------------------
# Upper Confidence Bound
# -----------------------------
# Moderate kappa because Function 7 has a promising current best but is still 6D.
kappa = 1.8
ucb = mean + kappa * std


# -----------------------------
# Filtered hybrid EI-UCB
# -----------------------------
# Keep candidates that are either near competitive predicted performance
# or in the strongest predicted group.
mean_filter = mean >= (y_best - 0.40)

if np.sum(mean_filter) == 0:
    mean_filter = mean >= np.percentile(mean, 90)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = ei[mean_filter]
filtered_ucb = ucb[mean_filter]

ei_norm = (filtered_ei - np.min(filtered_ei)) / (np.max(filtered_ei) - np.min(filtered_ei) + 1e-12)
ucb_norm = (filtered_ucb - np.min(filtered_ucb)) / (np.max(filtered_ucb) - np.min(filtered_ucb) + 1e-12)

# Slightly favour EI because Week 2 showed a strong promising region.
hybrid_score = 0.65 * ei_norm + 0.35 * ucb_norm

best_hybrid_index = np.argmax(hybrid_score)
query = filtered_candidates[best_hybrid_index]


# -----------------------------
# Output results
# -----------------------------
print("Acquisition used: Filtered Hybrid EI + UCB")
print("Number of candidates passing filter:", np.sum(mean_filter))
print("Suggested query:", query)

print("Portal format with hyphens:")
print("-".join(f"{v:.6f}" for v in query))

print("Portal format with x labels:")
print(",".join(f"x{i+1}:{v:.6f}" for i, v in enumerate(query)))

print("Predicted mean:", filtered_mean[best_hybrid_index])
print("Predicted std:", filtered_std[best_hybrid_index])
print("EI score:", filtered_ei[best_hybrid_index])
print("UCB score:", filtered_ucb[best_hybrid_index])
print("Hybrid score:", hybrid_score[best_hybrid_index])


Updated X shape: (32, 6)
Updated Y shape: (32,)
Best current x: [0.033169 0.329738 0.366358 0.234301 0.286315 0.704927]
Best current y: 2.4608579737515917
Fitted kernel: 0.954**2 * Matern(length_scale=[0.641, 0.257, 1, 0.631, 0.351, 0.464], nu=2.5) + WhiteKernel(noise_level=6.5e-10)
Acquisition used: Filtered Hybrid EI + UCB
Number of candidates passing filter: 5838
Suggested query: [0.         0.30009314 0.30481853 0.15805713 0.25246777 0.77658267]
Portal format with hyphens:
0.000000-0.300093-0.304819-0.158057-0.252468-0.776583
Portal format with x labels:
x1:0.000000,x2:0.300093,x3:0.304819,x4:0.158057,x5:0.252468,x6:0.776583
Predicted mean: 2.440289953907455
Predicted std: 0.1462708364406347
EI score: 0.04864557137411403
UCB score: 2.7035774595005977
Hybrid score: 0.9805782754927785


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
week3_x = np.array([[
    0.000000,
    0.300093,
    0.304819,
    0.158057,
    0.252468,
    0.776583
]])

week3_y = np.array([1.8105475948256518])

X = np.vstack([X, week1_x, week2_x, week3_x])
Y = np.append(Y, [
    week1_y[0],
    week2_y[0],
    week3_y[0]
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Shape:", X.shape, Y.shape)
print("Current best x:", best_x)
print("Current best y:", best_y)
print("Was Week 3 best?", best_index == len(Y) - 1)

Shape: (35, 6) (35,)
Current best x: [0.033169 0.329738 0.366358 0.234301 0.286315 0.704927]
Current best y: 2.4608579737515917
Was Week 3 best? False


In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)
import numpy as np

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(6, 0.2),
        length_scale_bounds=(1e-2, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-8,
        noise_level_bounds=(1e-10, 1e-3)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=35,
    random_state=42
)

gp.fit(X, Y)

print("Fitted kernel:", gp.kernel_)

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 42 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 29 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 24 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/

Fitted kernel: 0.714**2 * Matern(length_scale=[0.408, 2, 2, 0.16, 0.154, 2], nu=2.5) + WhiteKernel(noise_level=1e-10)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

In [7]:
trust_radius = np.full(6, 0.10)

lower_bounds = np.maximum(0, best_x - trust_radius)
upper_bounds = np.minimum(1, best_x + trust_radius)

bounds = list(zip(lower_bounds, upper_bounds))

print("Lower bounds:", lower_bounds)
print("Upper bounds:", upper_bounds)

Lower bounds: [0.       0.229738 0.266358 0.134301 0.186315 0.604927]
Upper bounds: [0.133169 0.429738 0.466358 0.334301 0.386315 0.804927]


In [8]:
kappa = 0.6

def negative_ucb(point):
    point = np.asarray(point).reshape(1, -1)

    mean, std = gp.predict(point, return_std=True)
    ucb = mean[0] + kappa * std[0]

    return -ucb

In [9]:
from scipy.optimize import minimize

rng = np.random.default_rng(42)

random_starts = rng.uniform(
    lower_bounds,
    upper_bounds,
    size=(180, 6)
)

starting_points = [
    best_x,
    np.clip(week3_x[0], lower_bounds, upper_bounds)
]

starting_points.extend(random_starts)

results = []

for start in starting_points:
    result = minimize(
        negative_ucb,
        x0=start,
        method="L-BFGS-B",
        bounds=bounds
    )

    if result.success:
        results.append(result)

if not results:
    raise RuntimeError("No successful optimisation runs")

best_result = min(results, key=lambda result: result.fun)
query = best_result.x

In [10]:
query_mean, query_std = gp.predict(
    query.reshape(1, -1),
    return_std=True
)

query_ucb = query_mean[0] + kappa * query_std[0]

distances = np.linalg.norm(X - query, axis=1)
nearest_index = np.argmin(distances)
nearest_distance = distances[nearest_index]

print("Method: Focused trust-region GP-UCB")
print("Suggested Week 4 query:", query)

print(
    "Portal format:",
    "-".join(f"{value:.6f}" for value in query)
)

print("Current best observed output:", best_y)
print("Predicted mean:", query_mean[0])
print("Predicted std:", query_std[0])
print("UCB:", query_ucb)
print("Distance to nearest observation:", nearest_distance)
print("Nearest observed point:", X[nearest_index])
print("Nearest observed output:", Y[nearest_index])

Method: Focused trust-region GP-UCB
Suggested Week 4 query: [0.         0.32141344 0.40036538 0.25832015 0.2850378  0.76274358]
Portal format: 0.000000-0.321413-0.400365-0.258320-0.285038-0.762744
Current best observed output: 2.4608579737515917
Predicted mean: 2.472010947119493
Predicted std: 0.0882686770771587
UCB: 2.5249721533657885
Distance to nearest observation: 0.07903980335986546
Nearest observed point: [0.033169 0.329738 0.366358 0.234301 0.286315 0.704927]
Nearest observed output: 2.4608579737515917
